In [174]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os
from TimeseriesSequence import TimeseriesSequence


# Create saved_models directory if it doesn't exist
os.makedirs('saved_models', exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.18.0
GPU Available: []


In [175]:
data_root = 'db/'
df = pd.read_csv(data_root + 'Extruder_data_seconds.csv')

In [176]:
df.columns

Index(['Timestamp', 'Temp Downspout', 'Temp Zone 1', 'Temp Zone 2',
       'Temp Zone 3', 'Temp Zone 4', 'Die Temp', 'Bin Feeder RPM',
       'Extruder Load', 'Extruder RPM', 'Knife RPM', 'Extruder Water',
       'Cylinder Water', 'Extruder Steam', 'Cylinder Steam', 'kW', 'Comment',
       'ts', 'Bulk Density', 'WHC', 'In-barrel MC', 'Time(s)', 'Time(min)',
       'SME-Load', 'SME-Watt', 'Raw Material Moisture', 'group'],
      dtype='object')

In [177]:
input_features = ['Bulk Density','Extruder Load',
    'Extruder RPM', 'Extruder Water','Cylinder Water', 'Temp Zone 3', 'Temp Zone 4',
    'Die Temp','SME-Watt']
output_features = ['Die Temp','SME-Watt']

In [178]:
independent_vars =['Raw Material Moisture','Extruder Water','Extruder RPM','Cylinder Water','Temp Zone 3', 'Temp Zone 4']
dependent_vars = ['Bulk Density','Extruder Load','Die Temp','SME-Watt']
data_control_vars = ['group','ts','Timestamp','Time(s)']

output_vars =['Die Temp','SME-Watt']

In [179]:
selected_features = independent_vars + dependent_vars + data_control_vars
input_features = independent_vars + output_vars
df = df[selected_features]

In [180]:
grouped_ts = df.groupby('group')
test_df = pd.DataFrame()
train_df = pd.DataFrame()
val_df = pd.DataFrame()
for name,group in grouped_ts:

    total_len = len(group)
    train_idx = int(0.7 * total_len)
    val_idx = int(0.8 * total_len)
    
    train_df_temp = group.iloc[:train_idx].copy()
    val_df_temp = group.iloc[train_idx:val_idx].copy()
    test_df_temp = group.iloc[val_idx:].copy()

    train_df = pd.concat([train_df, train_df_temp], ignore_index=True)
    val_df = pd.concat([val_df, val_df_temp], ignore_index=True) 
    test_df = pd.concat([test_df, test_df_temp], ignore_index=True)

train_df['Split'] = 'train'
val_df['Split'] = 'val' 
test_df['Split'] = 'test'

full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
full_df.to_csv(f"{data_root}LSTM_splited_df.csv")

In [181]:
x_train = train_df[input_features].values
y_train = train_df[output_vars].values

x_val = val_df[input_features].values
y_val = val_df[output_vars].values

x_test = test_df[input_features].values
y_test = test_df[output_vars].values

input_scaler = MinMaxScaler()
output_scaler = MinMaxScaler()


x_train_scaled = input_scaler.fit_transform(x_train).astype(np.float32)
y_train_scaled = output_scaler.fit_transform(y_train).astype(np.float32)

x_val_scaled = input_scaler.transform(x_val).astype(np.float32)
y_val_scaled = output_scaler.transform(y_val).astype(np.float32)

x_test_scaled = input_scaler.transform(x_test).astype(np.float32)
y_test_scaled = output_scaler.transform(y_test).astype(np.float32)

In [182]:
batch_size = 64
window = 60
horizon = 30
timestep = 1

In [183]:
train_gen = TimeseriesSequence(
    x=x_train_scaled,
    y=y_train_scaled,
    window_size=window,
    prediction_horizon=horizon,
    timestep=timestep,
    batch_size=batch_size,
    shuffle=False
)

val_gen = TimeseriesSequence(
    x=x_val_scaled,
    y=y_val_scaled,
    window_size=window,
    prediction_horizon=horizon,
    timestep=timestep,
    batch_size=batch_size,
    shuffle=False
)

test_gen = TimeseriesSequence(
    x=x_test_scaled,
    y=y_test_scaled,
    window_size=window,
    prediction_horizon=horizon,
    timestep=timestep,
    batch_size=batch_size,
    shuffle=False
)

In [184]:
# Define model parameters
n_features = len(input_features)
n_outputs = len(output_vars)
hidden_dim = 64

inputs = keras.Input(shape=(window, n_features))
lstm1 = layers.LSTM(hidden_dim, return_sequences=True)(inputs)
lstm2 = layers.LSTM(hidden_dim//2)(lstm1)
dense = layers.Dense(n_outputs * horizon)(lstm2)
output = layers.Reshape((horizon, n_outputs))(dense)

model = Model(inputs, output)

model.summary()


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 60, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 60, 64)         │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_14 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 60)             │         1,980 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_5 (Reshape)             │ (None, 30, 2)          │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,084 (129.23 KB)

 Trainable params: 33,084 (129.23 KB)

 Non-trainable params: 0 (0.00 B)

In [185]:
lr = 0.0010
optimizer = keras.optimizers.Adam(learning_rate=lr)

model.compile(optimizer=optimizer, loss='mse',metrics=['mae'])

In [186]:
early_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model_checkpoint_callback = ModelCheckpoint(
    filepath='saved_models/best_lstm_model.keras',
    save_weights_only=False,
    monitor='val_loss',
    mode='min',
    save_best_only=True)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    batch_size=batch_size,
    callbacks=[early_callback, model_checkpoint_callback]
)

Epoch 1/30


c:\Users\amirbg\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - loss: 0.0826 - mae: 0.2077 - val_loss: 0.0134 - val_mae: 0.0836
Epoch 2/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - loss: 0.0098 - mae: 0.0713 - val_loss: 0.0104 - val_mae: 0.0652
Epoch 3/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss: 0.0079 - mae: 0.0606 - val_loss: 0.0115 - val_mae: 0.0774
Epoch 4/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss: 0.0077 - mae: 0.0608 - val_loss: 0.0117 - val_mae: 0.0765
Epoch 5/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss: 0.0068 - mae: 0.0555 - val_loss: 0.0099 - val_mae: 0.0610
Epoch 6/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss: 0.0066 - mae: 0.0546 - val_loss: 0.0101 - val_mae: 0.0681
Epoch 7/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss: 0.0065 - mae: 0.0541 - val_loss: 0.0104 - val_mae: 0.0657
Epoch 8/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss: 0.0060 - mae: 0.0507 - val_loss: 0.0104 - val_mae: 0.0721
Epoch 9/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss

In [187]:
history_df = pd.DataFrame(history.history)
history_df.to_csv('training_hists/LSTM_FC_history_1.csv', index=False)

In [188]:
history_df

,loss,mae,val_loss,val_mae
0,0.082620,0.207722,0.013437,0.083579
1,0.009848,0.071340,0.010390,0.065222
2,0.007928,0.060617,0.011511,0.077426
3,0.007674,0.060821,0.011750,0.076519
4,0.006807,0.055526,0.009894,0.060951
5,0.006636,0.054556,0.010113,0.068097
6,0.006548,0.054067,0.010408,0.065688
7,0.006003,0.050688,0.010436,0.072089
8,0.006064,0.050889,0.009352,0.061174
9,0.005257,0.045385,0.008359,0.050624


In [189]:
best_model = keras.models.load_model('saved_models/best_lstm_model.keras')

In [ ]:
combined_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

combined_df['Die Temp_pred'] = combined_df['Die Temp'].copy()
combined_df['SME-Watt_pred'] = combined_df['SME-Watt'].copy()

def smooth_predictions(pred, window_size=30):
    if len(pred) < window_size:
        return pred
    smoothed = np.copy(pred)
    for i in range(len(pred)):
        start = max(0, i - window_size // 2)
        end = min(len(pred), i + window_size // 2 + 1)
        smoothed[i] = np.mean(pred[start:end], axis=0)
    return smoothed

for group_name in combined_df['group'].unique():
    group_mask = combined_df['group'] == group_name
    group_data = combined_df[group_mask].copy()
    group_indices = combined_df[group_mask].index  
    
    x_group = group_data[input_features].values
    y_group = group_data[output_vars].values
    
    x_group_scaled = input_scaler.transform(x_group).astype(np.float32)
    y_group_scaled = output_scaler.transform(y_group).astype(np.float32)
    

    idx = 0
    while idx + window + horizon <= len(x_group_scaled):
        x_window = x_group_scaled[idx:idx + window]
        x_window = x_window.reshape(1, window, -1)
        
        pred_scaled = best_model.predict(x_window, verbose=0)
        
        pred = output_scaler.inverse_transform(pred_scaled.reshape(-1, n_outputs))
        
        if horizon > 1:
            pred = smooth_predictions(pred, window_size=max(5, horizon))
        
        for h in range(horizon):
            pred_idx = idx + window + h
            if pred_idx < len(group_indices):
                original_idx = group_indices[pred_idx]
                combined_df.loc[original_idx, 'Die Temp_pred'] = pred[h, 0]
                combined_df.loc[original_idx, 'SME-Watt_pred'] = pred[h, 1]
        
        idx += horizon



In [193]:
combined_df.to_csv(f"simulations/LSTM_predictions_v2.csv", index=False)